# Parameters
## Main parameters

In [1]:
cell_type    = 'CD4'
learn_rate   = 1e-3
batch_size   = 32
test_study   = 'Chow_2023'

In [2]:
# Parameters
cell_type = "CD4"
learn_rate = 0.001
batch_size = 32
test_study = "Liu_2025"


## Additional parameters

In [3]:
data_dir         = "/Users/wsun/research/CAT/data/"
results_dir      = "classification_result/"
max_epochs       = 20
model_patience   = 10
validating_frac  = 0.10
classify_layers  = []
encode_layers    = [128 , 32]
bottle_size      = 8


# Load modules and define functions
## Imports

In [4]:
import os
import argparse
import itertools
import random
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp

from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc

import tensorflow as tf
import keras
from keras import optimizers
from keras.layers import Input, Dense, Reshape, Flatten, Dropout
from keras.models import Model
from keras.callbacks import EarlyStopping, LearningRateScheduler

import math
import numpy as np
import pandas as pd
import matplotlib
np.random.seed(42)
tf.random.set_seed(42)
matplotlib.use("Agg")
import matplotlib.pyplot as plt


## Functions

### Plot classification function and save classification results

In [5]:
def plot_classification(m1, start, plot_path):
    # plot the training loss and accuracy
    end = len(m1.history['classification_accuracy'])
    N   = np.arange(start, end)
    s   = slice(start,end)

    plt.style.use("ggplot")
    plt.figure(figsize=(4, 3), dpi=300)

    plt.plot(N, (m1.history["classification_accuracy"][s]), label="train_accuracy")
    plt.plot(N, (m1.history["val_classification_accuracy"][s]), label="val_accuracy")

    plt.xlabel("Epoch #")
    plt.ylabel("Classification Accuracy")
    plt.legend()
    plt.subplots_adjust(left=0.25, right=0.98, top=0.98, bottom=0.2)
    plt.savefig(plot_path)
    plt.close()


def table_save_classification(m1, start, file_path):
    end = len(m1.history['classification_accuracy'])
    epochs = range(start, end)
    train_losses = m1.history['classification_accuracy'][start:end]
    val_losses = m1.history['val_classification_accuracy'][start:end]

    # Write the table to the file
    with open(file_path, 'w') as f:
        f.write("Epoch\tTrain_Acc\tVal_Acc\n")  # Header row
        for epoch, train_loss, val_loss in zip(epochs, train_losses, val_losses):
            f.write(f"{epoch}\t{train_loss:.4f}\t{val_loss:.4f}\n")


# Define model
## Model configuration

In [6]:
config = 'en_'
for size in encode_layers:
  config += str(size) + '_'

config += 'bn_' + str(bottle_size) + '_cl_'

for size in classify_layers:
  config += str(size) + '_'


config = config + 'bs_' + str(batch_size) + '_lr_' + str(learn_rate)
config = config + '_e_' + str(max_epochs)
config

'en_128_32_bn_8_cl_bs_32_lr_0.001_e_20'

## Read in data

In [7]:
adata = sc.read_h5ad(data_dir + cell_type + "_combined_filtered.h5ad")

print("Count Data:")
print(adata.X.shape)
print(adata.X[:5,:4])
print()
print("Meta Data:")
print(adata.obs.shape)
pd.set_option('display.max_columns', None)  # show all columns
print(adata.obs[:5])
print()

Count Data:
(346201, 6245)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (5, 4)>
  Coords	Values
  (2, 1)	1.0
  (4, 3)	1.0

Meta Data:
(346201, 44)
                                                   study2  n_genes_by_counts  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021_CD4    Zheng_2021_CD4               1346   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024_CD4    Chen_2024_CD4               1029   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024_CD4   Chen_2024_CD4               1236   
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023_CD4    Chow_2023_CD4                971   
P47-TCAGCTCGTTCAGCGC-1-Liu_2025_CD4          Liu_2025_CD4               1041   

                                           total_counts         TRB_cdr3  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021_CD4          3215.0  CA*MRGFIMATPSVR   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024_CD4         2930.0    CAAAATNNNEQFF   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024_CD4        3943.0     CAAAGAGTEAFF   
CGCTTCAAGGCAAAGA-1_PEM1

In [8]:
depth_col = "total_counts"        # or your total count column
print("Depth Column:", depth_col)
print(adata.obs[depth_col].describe())

Depth Column: total_counts
count    346201.000000
mean       3761.304589
std        1635.527652
min         458.000000
25%        2641.000000
50%        3519.000000
75%        4569.000000
max       19224.000000
Name: total_counts, dtype: float64


In [9]:
from scipy.sparse import issparse
print("is sparse:", issparse(adata.X))

adata.layers["counts"] = adata.X.copy()

target_sum = 3000
d = adata.obs[depth_col].to_numpy()

scale = np.divide(target_sum, d, out=np.zeros_like(d, dtype=float), where=d>0)

if issparse(adata.X):
    X = adata.X.tocsr()
    adata.layers["norm"] = X.multiply(scale[:, None]).tocsr()
else:
    adata.layers["norm"] = adata.X * scale[:, None]

adata.X = adata.layers["norm"].copy()

sc.pp.log1p(adata)
print(adata.X.shape)
print(adata.X[:2, :4])

is sparse: True


(346201, 6245)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 0 stored elements and shape (2, 4)>


## Exploratory analysis

This analysis is the same for any testing data and thus only need to do it for the first test data. 

In [10]:
if(test_study == "Chen_2024"):
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata_hvg = adata[:, adata.var["highly_variable"]].copy()
    sc.tl.pca(adata_hvg, n_comps=50, svd_solver="arpack", zero_center=False)
    sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30, use_rep="X_pca")
    sc.tl.umap(adata_hvg, min_dist=0.3)

    sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(6, 5))

    for col in [f"pos_score_{cell_type}", f"neg_score_{cell_type}"]:
        # copy the obs columns over if we sliced to HVGs (obs is preserved, so this is optional)
        assert col in adata_hvg.obs
        sc.pl.umap(
            adata_hvg,
            color=col,
            color_map="viridis",   # try "plasma", "magma", "turbo" if you prefer
            edges=False,
            frameon=False,
            save=f"figures/umap_{col}.png"
        )

    sc.pl.umap(
        adata_hvg,
        color="study",
        edges=False,
        frameon=False,
        save=f"figures/umap_study_{cell_type}.png"
    )

    sc.pl.umap(
        adata_hvg,
        color="Tissue",
        edges=False,
        frameon=False,
        save=f"figures/umap_tissue_{cell_type}.png"
    )



## Try to classify cancer reactive or not using postive vs. negative score

In [11]:
print(pd.crosstab(adata.obs["study"], adata.obs["label"]))

n_samples = ((adata.obs["study"] == test_study) & (adata.obs["label"] == 1)).sum()
print(f"Number of samples: {n_samples}")

if n_samples == 0:
    print("No samples from the test study.")
    exit(0)

label            0      1
study                    
Chen_2024   109237    465
Chow_2023    68218      0
Liu_2022     20242   2855
Liu_2025     96904  12585
Zheng_2021   32162   3533
Number of samples: 12585


In [12]:
from pandas.api.types import is_numeric_dtype
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc

pos_score_col = f"pos_score_{cell_type}"
neg_score_col = f"neg_score_{cell_type}"

req_cols = [pos_score_col, neg_score_col, 'label', 'study']
missing = [c for c in req_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

df = adata.obs[req_cols].copy().dropna(subset=req_cols)

# label must be numeric and only {0,1}
if not is_numeric_dtype(df['label']):
    raise ValueError(f"`label` must be numeric (0/1). Got dtype={df['label'].dtype}")
vals = set(np.unique(df['label'].values))
if not vals.issubset({0, 1}):
    raise ValueError(f"`label` must contain only 0 and 1. Found values: {sorted(vals)}")

# ---- 2) Split by study (test = test_study)
test_mask = (df['study'].astype(str) == test_study)
if not test_mask.any():
    raise ValueError(f"No rows with study == {test_study} for testing.")
if (~test_mask).sum() == 0:
    raise ValueError(f"All rows are {test_study}; need non–{test_study} rows for training.")

X_train = df.loc[~test_mask, [pos_score_col, neg_score_col]].to_numpy()
y_train = df.loc[~test_mask, 'label'].astype(int).to_numpy()

X_test  = df.loc[test_mask,  [pos_score_col, neg_score_col]].to_numpy()
y_test  = df.loc[test_mask,  'label'].astype(int).to_numpy()

# Both classes must be present in train; ROC needs positives in test
if np.unique(y_train).size < 2:
    raise ValueError("Training set must contain both classes 0 and 1.")
if (y_test == 1).sum() == 0 or (y_test == 0).sum() == 0:
    raise ValueError("Test set must contain both classes 0 and 1 to compute ROC/AUC.")

# ---- 3) Model: scale -> logistic regression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, class_weight='balanced', solver='lbfgs')),
])
pipe.fit(X_train, y_train)

# ---- 4) Predict + ROC/AUC 
y_score = pipe.predict_proba(X_test)[:, 1]  # prob of class 1
fpr, tpr, _ = roc_curve(y_test, y_score, pos_label=1)
roc_auc = auc(fpr, tpr)
print(f"AUC on {test_study}: {roc_auc:.4f}")

# ---- 5) Plot ROC
plt.figure(figsize=(5, 4), dpi=300)
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC — Logistic Regression (test: {test_study})')
plt.legend(loc='lower right')
plt.subplots_adjust(left=0.2)  # default is ~0.125
plt.savefig(f"figures/logistic_regression_roc_{cell_type}_{test_study}.png")

AUC on Liu_2025: 0.9665


In [13]:
adata_sub = adata[adata.obs["study"] != test_study].copy()
adata_sub.obs["label"] = pd.Categorical(adata_sub.obs["label"].astype(str),
                                        categories=["0", "1"])

# (optional) sanity checks
assert (adata_sub.obs["label"] == "0").sum() > 0 and (adata_sub.obs["label"] == "1").sum() > 0
print(adata_sub.obs["label"].value_counts(dropna=False))

label
0    229859
1      6853
Name: count, dtype: int64


In [14]:
# -----------------------------
# 1) Rank-sum DE: label==1 vs label==0
# -----------------------------
sc.tl.rank_genes_groups(
    adata_sub,
    groupby="label",
    groups=["1"],             # test group(s)
    reference="0",            # control group
    method="wilcoxon",
    corr_method="benjamini-hochberg",  # FDR
    use_raw=False,            # set True if you saved counts in .raw and want to test on raw
    n_genes=adata_sub.n_vars, # compute stats for all genes
    key_added="de_wilcoxon"
)

# -----------------------------
# 2) Collect results into a DataFrame
# -----------------------------
# For Scanpy >=1.9:
df_de = sc.get.rank_genes_groups_df(adata_sub, group="1", key="de_wilcoxon")
# Columns: ['names','scores','logfoldchanges','pvals','pvals_adj','pts','pts_rest']

# Sort by adjusted p-value (FDR)
df_de = df_de.sort_values("pvals_adj", ascending=True)

# Save full results and top 1000
df_de.to_csv(f"DE_results/DE_{cell_type}_test_study_{test_study}.csv", index=False)

# -----------------------------
# 3) Volcano plot
# -----------------------------
x = df_de["logfoldchanges"]                 # log2FC in Scanpy is natural-log FC; if you prefer log2, convert:
# If you want true log2 fold-change:
# x = df_de["logfoldchanges"] / np.log(2)

y = -np.log10(df_de["pvals_adj"].clip(lower=1e-300))

plt.figure(figsize=(6,5), dpi=150)
plt.scatter(x, y, s=8, alpha=0.5)

# Highlight top 1000 by FDR
is_top = df_de.index.isin(df_de.index[:1000])
plt.scatter(x[is_top], y[is_top], s=10, alpha=0.8)

# Reference lines (optional)
plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)
plt.axvline(0, linestyle=":", linewidth=1)

plt.xlabel("log fold change (1 vs 0)")      # note: natural log by default
plt.ylabel("-log10(FDR)")
plt.title(f"Wilcoxon DE: (excluding {test_study})")
plt.tight_layout()
plt.savefig(f"figures/volcano_{cell_type}_test_study_{test_study}.png")
plt.close()


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


### Prepare data splitting indices

In [15]:
print(test_study)
print(adata.obs.shape)
print(adata.obs['study'].value_counts())

print(adata.obs["label"].value_counts())
print(pd.crosstab(adata.obs["study"], adata.obs["label"]))

test_indices = np.where(adata.obs['study'] == test_study)[0]
train_val_indices = np.where(adata.obs['study'] != test_study)[0]
train_indices, val_indices = train_test_split(train_val_indices, test_size=validating_frac, 
                                              stratify=adata.obs['study'][train_val_indices], 
                                              random_state=42)
print("Length of train_indices:", len(train_indices))
print("Length of validation_indices:", len(val_indices))
print("Length of test_indices:", len(test_indices))

Liu_2025
(346201, 44)
study
Chen_2024     109702
Liu_2025      109489
Chow_2023      68218
Zheng_2021     35695
Liu_2022       23097
Name: count, dtype: int64
label
0    326763
1     19438
Name: count, dtype: int64
label            0      1
study                    
Chen_2024   109237    465
Chow_2023    68218      0
Liu_2022     20242   2855
Liu_2025     96904  12585
Zheng_2021   32162   3533
Length of train_indices: 213040
Length of validation_indices: 23672
Length of test_indices: 109489


/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/1704601559.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stratify=adata.obs['study'][train_val_indices],


### Split traning test data

In [16]:
top_genes = df_de.sort_values("pvals").head(1000)["names"].tolist()
print(top_genes[:10])

adata_top1000 = adata[:, top_genes]
X = adata_top1000.X.toarray()
print(X.shape)

['DUSP4', 'CACYBP', 'DNPH1', 'CREM', 'UBB', 'DNAJA1', 'HLA-DPA1', 'RHOA', 'IL2RB', 'TNFRSF1B']


(346201, 1000)


In [17]:
# Split the data into training, validation, and testing sets
trainX = X[train_indices]
valX = X[val_indices]
testX = X[test_indices]

trainY = adata.obs["label"][train_indices]
valY   = adata.obs["label"][val_indices]
testY  = adata.obs["label"][test_indices]

print("Training Data:")
print(trainX.shape)
print(trainY.shape)
print()

print("Validation Data:")
print(valX.shape)
print(valY.shape)
print()

print("Testing Data:")
print(testX.shape)
print(testY.shape)
print()

print('trainX[0:2, 0:4]:')
print(trainX[0:2, 0:4])
print()

print('trainY[0:2]:')
print(trainY[0:2])
print()

print('valX[0:2, 0:4]:')
print(valX[0:2, 0:4])
print()

print('valY[0:2]:')
print(valY[0:2])
print()

print('testX[0:2, 0:4]:')
print(testX[0:2, 0:4])
print()

print('testY[0:2]:')
print(testY[0:2])
print()


Training Data:
(213040, 1000)
(213040,)

Validation Data:
(23672, 1000)
(23672,)

Testing Data:
(109489, 1000)
(109489,)

trainX[0:2, 0:4]:
[[0.         0.73646244 0.         1.4505834 ]
 [0.         0.54451002 0.         0.        ]]

trainY[0:2]:
CRC01-T-IV_TACAGTGGTTCGGGCT-Chen_2024_CD4    0
CRC12-B-II_ACTTTCACAGTAAGAT-Chen_2024_CD4    0
Name: label, dtype: int64

valX[0:2, 0:4]:
[[0.         0.50012615 0.50012615 0.        ]
 [0.97154988 0.59939572 0.97154988 2.02394793]]

valY[0:2]:
CRC12-B-I_AAGGCAGTCCGAGCCA-Chen_2024_CD4    0
P30.ut.AGATCTGAGTGTACCT-1-Liu_2022_CD4      1
Name: label, dtype: int64

testX[0:2, 0:4]:
[[0.         1.09816804 0.         0.        ]
 [0.         0.         0.35290143 1.47830977]]

testY[0:2]:
P47-TCAGCTCGTTCAGCGC-1-Liu_2025_CD4     0
P485-AGGTCATGTAAGAGAG-1-Liu_2025_CD4    0
Name: label, dtype: int64



/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/3945436047.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  trainY = adata.obs["label"][train_indices]
/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/3945436047.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  valY   = adata.obs["label"][val_indices]
/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/3945436047.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by 

## Set up model

In [18]:
# Parameters
input_dim = trainX.shape[1]
encoding_dim = bottle_size  # Bottleneck layer size

# Input layer
input = Input(shape=(input_dim,))

# Encoder
encoded = Dense(encode_layers[0], activation='relu')(input)
for i in range(len(encode_layers)-1):
  encoded = Dense(encode_layers[i+1], activation='relu')(encoded)
bottleneck = Dense(encoding_dim, activation='relu')(encoded)  # Latent space (bottleneck)

# Decoder
decoded = Dense(32, activation='relu')(bottleneck)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(input_dim, activation='relu', name='reconstruction')(decoded)  # Output layer (reconstructed input)

# Classification layer on top of the bottleneck
if len(classify_layers)==0:
  classifier_output = Dense(1, activation='sigmoid',name='classification')(bottleneck)
else:
  layer = Dense(size, activation='relu')(bottleneck)
  for i in range(len(classify_layers)-1):
    layer = Dense(classify_layers[i+1], activation='relu')(layer)
  classifier_output = Dense(1, activation='sigmoid',name='classification')(layer)

# Full model (Autoencoder + Classification)
full_model = Model(inputs = input, outputs = {"reconstruction": decoded, "classification": classifier_output})
full_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    128,128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      4,128 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        264 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │        288 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │      4,224 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification      │ (None, 1)         │          9 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reconstruction      │ (None, 1000)      │    129,000 │ dense_4[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 266,041 (1.01 MB)

 Trainable params: 266,041 (1.01 MB)

 Non-trainable params: 0 (0.00 B)

# Train the model

In [19]:
reconstruction_weight = 1.0  # Weight for reconstruction loss
classification_weight = 1.0  # Weight for classification loss

In [20]:
# Compile the model
def scheduler(epoch, lr):
    if epoch < 10:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-0.1))

lr_scheduler = LearningRateScheduler(scheduler)

adam1 = optimizers.Adam(learning_rate=learn_rate)

full_model.compile(
	optimizer=adam1,
	loss={'reconstruction':'mean_absolute_error',
	      'classification':'binary_crossentropy'},  # Two losses: MAE for reconstruction, BCE for classification
	metrics={'reconstruction': 'mean_absolute_error','classification': 'accuracy'}, 
	loss_weights = {
			 'reconstruction':reconstruction_weight,
			 'classification':classification_weight
             }
)

#early stopping
early_stopping = EarlyStopping(
    monitor='val_classification_accuracy',  # Metric to monitor
    patience=model_patience,                # Number of epochs to wait
    restore_best_weights=True, 
    mode='max'
)

# Train the model

history = full_model.fit(x = trainX, y = {"reconstruction": trainX, "classification": trainY}, epochs=max_epochs, 
                         batch_size=batch_size, verbose=2,
                         validation_data=(valX, {"reconstruction": valX, "classification": valY}),
                         callbacks=[lr_scheduler, early_stopping])


Epoch 1/20


6658/6658 - 13s - 2ms/step - classification_accuracy: 0.9905 - classification_loss: 0.0247 - loss: 0.2815 - reconstruction_loss: 0.2568 - reconstruction_mean_absolute_error: 0.2568 - val_classification_accuracy: 0.9906 - val_classification_loss: 0.0217 - val_loss: 0.2617 - val_reconstruction_loss: 0.2400 - val_reconstruction_mean_absolute_error: 0.2400 - learning_rate: 0.0010


Epoch 2/20


6658/6658 - 12s - 2ms/step - classification_accuracy: 0.9920 - classification_loss: 0.0201 - loss: 0.2577 - reconstruction_loss: 0.2376 - reconstruction_mean_absolute_error: 0.2376 - val_classification_accuracy: 0.9909 - val_classification_loss: 0.0210 - val_loss: 0.2578 - val_reconstruction_loss: 0.2368 - val_reconstruction_mean_absolute_error: 0.2368 - learning_rate: 0.0010


Epoch 3/20


6658/6658 - 13s - 2ms/step - classification_accuracy: 0.9927 - classification_loss: 0.0182 - loss: 0.2542 - reconstruction_loss: 0.2359 - reconstruction_mean_absolute_error: 0.2359 - val_classification_accuracy: 0.9912 - val_classification_loss: 0.0213 - val_loss: 0.2547 - val_reconstruction_loss: 0.2335 - val_reconstruction_mean_absolute_error: 0.2334 - learning_rate: 0.0010


... (30 outputs left out)

Epoch 19/20


6658/6658 - 12s - 2ms/step - classification_accuracy: 0.9990 - classification_loss: 0.0027 - loss: 0.2254 - reconstruction_loss: 0.2227 - reconstruction_mean_absolute_error: 0.2227 - val_classification_accuracy: 0.9910 - val_classification_loss: 0.0631 - val_loss: 0.2861 - val_reconstruction_loss: 0.2231 - val_reconstruction_mean_absolute_error: 0.2230 - learning_rate: 4.0657e-04


Epoch 20/20


6658/6658 - 12s - 2ms/step - classification_accuracy: 0.9993 - classification_loss: 0.0021 - loss: 0.2247 - reconstruction_loss: 0.2226 - reconstruction_mean_absolute_error: 0.2226 - val_classification_accuracy: 0.9912 - val_classification_loss: 0.0643 - val_loss: 0.2872 - val_reconstruction_loss: 0.2229 - val_reconstruction_mean_absolute_error: 0.2229 - learning_rate: 3.6788e-04


In [21]:
name = cell_type
name += "_" + test_study
name += "_learn_rate_" + str(learn_rate)
name += "_batch_size_" + str(batch_size)
name += "_cla_weight_" + str(classification_weight)
name += "_rec_weight_" + str(reconstruction_weight)
if model_patience >= max_epochs:
  name += "_epochs_" + str(max_epochs)
else:
  name += "_max_epochs_" + str(max_epochs)

print(name)

path = os.path.join(results_dir, name)
if not os.path.exists(path):
  os.makedirs(path)


CD4_Liu_2025_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20


In [22]:
plt.style.use("ggplot")
plt.figure(figsize=(4, 3), dpi=300)

plt.plot(history.history['reconstruction_loss'], label='Reconstruction (MAE)', color='tab:blue')
plt.plot(history.history['classification_loss'], label='Classification (BCE)', color='tab:orange')

plt.xlabel("Epoch #")
plt.ylabel("Loss")
plt.legend()
plt.subplots_adjust(left=0.2, right=0.98, top=0.98, bottom=0.2)
plt.savefig(f"{path}/classification_loss_plot.png")
plt.close()

plt.style.use("ggplot")
plt.figure(figsize=(4, 3), dpi=300)
plt.plot(history.history['val_reconstruction_loss'], label='Reconstruction (MAE)', color='tab:blue')
plt.plot(history.history['val_classification_loss'], label='Classification (BCE)', color='tab:orange')

plt.xlabel("Epoch #")
plt.ylabel("Loss")
plt.legend()
plt.subplots_adjust(left=0.2, right=0.98, top=0.98, bottom=0.2)
plt.savefig(f"{path}/val_classification_loss_plot.png")
plt.close()

plot_classification(history, 1, f"{path}/classification_accuracy_plot.png")
table_save_classification(history, 1, f"{path}/classification_accuracy_table.txt")


# Make predictin using test data

In [23]:
prediction = full_model.predict(testX)["classification"].flatten()
print(testY)

# Create a DataFrame with two columns: 'Prediction' and 'True Label'
df = pd.DataFrame({
    'Prediction': prediction,
    'True Label': testY
})

# Show the DataFrame
print(df)


   1/3422 ━━━━━━━━━━━━━━━━━━━━ 1:36 28ms/step

 111/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 457us/step 

 232/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 435us/step

 233/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 435us/step

 343/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 442us/step

 458/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 441us/step

... (25 outputs left out)

3234/3422 ━━━━━━━━━━━━━━━━━━━━ 0s 420us/step

3354/3422 ━━━━━━━━━━━━━━━━━━━━ 0s 420us/step

3422/3422 ━━━━━━━━━━━━━━━━━━━━ 1s 425us/step


P47-TCAGCTCGTTCAGCGC-1-Liu_2025_CD4     0
P485-AGGTCATGTAAGAGAG-1-Liu_2025_CD4    0
P579-GCTTGAAGTAAGTTCC-1-Liu_2025_CD4    0
P587-GGGATGAGTCGGCACT-1-Liu_2025_CD4    0
P258-TACCTATCATACTACG-1-Liu_2025_CD4    0
                                       ..
P166-AACACGTAGTGTGAAT-1-Liu_2025_CD4    0
P125-CATGACATCGAATCCA-1-Liu_2025_CD4    0
P277-CAAGAAATCTGTCTAT-1-Liu_2025_CD4    0
P279-GAAATGATCTGCCCTA-1-Liu_2025_CD4    0
P226-GTAACGTTCACATGCA-1-Liu_2025_CD4    0
Name: label, Length: 109489, dtype: int64
                                        Prediction  True Label
P47-TCAGCTCGTTCAGCGC-1-Liu_2025_CD4   2.037226e-24           0
P485-AGGTCATGTAAGAGAG-1-Liu_2025_CD4  4.128203e-13           0
P579-GCTTGAAGTAAGTTCC-1-Liu_2025_CD4  2.203064e-13           0
P587-GGGATGAGTCGGCACT-1-Liu_2025_CD4  4.630848e-14           0
P258-TACCTATCATACTACG-1-Liu_2025_CD4  3.488638e-12           0
...                                            ...         ...
P166-AACACGTAGTGTGAAT-1-Liu_2025_CD4  1.178577e-03     

## Summarize results
### Boxplot

In [24]:

plt.figure(figsize=(3, 3), facecolor='white')
sns.boxplot(x='True Label', y='Prediction', data=df)
plt.subplots_adjust(left=0.25, bottom=0.2)

ax = plt.gca()
ax.set_facecolor('none')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')

plt.savefig(f'{path}/test_box_plot_predictions.pdf', format='pdf')
plt.show()

df.to_csv(f"{path}/predictions_test_data.txt", sep='\t', index=True)


/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/1930487193.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ROC curve

In [25]:
# Calculate the ROC curve
fpr, tpr, _ = roc_curve(df['True Label'], df['Prediction'])

# Calculate the AUC
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(3, 3), facecolor='white')
plt.plot(fpr, tpr, color='orange', lw=2)
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([-0.02, 1.0])
plt.ylim([-0.02, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'AUC = {roc_auc:.3f}')

ax = plt.gca()
ax.set_facecolor('none')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')

plt.subplots_adjust(left=0.25, bottom=0.2)

# Save the plot
plt.savefig(f"{path}/roc_curve.pdf", format='pdf')

# Show the plot
plt.show()

/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_51447/2507533787.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
